# Feature Engineering for Net Flow Prediction (V3 - Cyclic + Station Trend)

This notebook builds a **versioned feature store (Gold V2)** designed for direct modeling of station-level net flow in a bike-sharing system.

The objective is to transform enriched spatiotemporal data into a **machine learning-ready dataset** that captures:

- Temporal demand patterns (hour, day of week, month)
- Short-term dynamics (recent net flow behavior)
- Station-specific trends
- Weather influence
- Impact of public events

This dataset serves as the **core input for both short-term (1-hour ahead) and multi-hour forecasting models**.

## Process Overview

The feature engineering pipeline follows these steps:

1. **Load Gold V2 spatiotemporal dataset**
   - Includes station activity, weather, and event features

2. **Filter to Downtown Toronto area**
   - Focus on high-density demand zones

3. **Create temporal features**
   - Hour, day-of-week, and month transformations
   - Cyclic encoding using sine/cosine functions

4. **Define target variable**
   - Net flow = arrivals − departures

5. **Generate lag-based features**
   - Short-term (1h, 2h)
   - Daily (24h)
   - Weekly (168h)

6. **Compute rolling statistics**
   - Mean (3h window)
   - Standard deviation (24h window)

7. **Build station-level behavioral features**
   - Recent activity trends (mean, variability, intensity)

8. **Remove incomplete rows**
   - Ensures model stability and consistency

9. **Write feature store**
   - Partitioned and versioned for reproducibility

In [0]:
# ============================================================
# BUILD GOLD_V2 FEATURES FOR NET FLOW - V3 CYCLIC STATIONTREND
# Creates a new versioned feature store for direct net_flow modeling
# Adds cyclic time features + station recent activity features
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------
GOLD_V2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"

FEATURES_NETFLOW_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/"
    "goldv2_features_netflow_v3_cyclic_stationtrend"
)

REBUILD_FEATURES = True

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

PI = 3.141592653589793

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

# ------------------------------------------------------------
# 3) VALIDATE INPUT
# ------------------------------------------------------------
if not path_exists(GOLD_V2_DIR):
    raise Exception(f"GOLD_V2_DIR not found: {GOLD_V2_DIR}")

if not REBUILD_FEATURES:
    raise Exception("Set REBUILD_FEATURES=True to build the new netflow feature store.")

print("Reading GOLD_V2 from:", GOLD_V2_DIR)

# ------------------------------------------------------------
# 4) READ GOLD_V2
# ------------------------------------------------------------
df = spark.read.parquet(GOLD_V2_DIR)

required_cols = {
    "station_id", "year", "month", "day", "hour",
    "departures", "arrivals",
    "lat", "lon",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby",
    "event_impact_score"
}

missing = sorted(list(required_cols - set(df.columns)))
if missing:
    raise Exception(f"GOLD_V2 is missing required columns: {missing}")

# ------------------------------------------------------------
# 5) FILTER DOWNTOWN ONLY
# ------------------------------------------------------------
df = df.filter(
    (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
    (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
)

stations_count = df.select("station_id").distinct().count()
rows_count = df.count()

print("Downtown stations:", stations_count)
print("Downtown rows    :", f"{rows_count:,}")

# ------------------------------------------------------------
# 6) BUILD BASE TIME COLUMNS
# ------------------------------------------------------------
df = (
    df
    .withColumn("date", F.make_date("year", "month", "day"))
    .withColumn("dow_num", F.dayofweek("date"))   # Spark: Sun=1 ... Sat=7
    .withColumn("is_weekend", F.when(F.col("dow_num").isin([1, 7]), 1).otherwise(0))
)

# ------------------------------------------------------------
# 6.1) CYCLIC TIME FEATURES
# ------------------------------------------------------------
df = (
    df
    # hour: 0..23
    .withColumn("hour_angle", F.lit(2.0 * PI) * F.col("hour") / F.lit(24.0))
    .withColumn("hour_sin", F.sin("hour_angle"))
    .withColumn("hour_cos", F.cos("hour_angle"))

    # dow_num: Spark Sun=1..Sat=7
    .withColumn("dow_idx", F.col("dow_num") - F.lit(1))
    .withColumn("dow_angle", F.lit(2.0 * PI) * F.col("dow_idx") / F.lit(7.0))
    .withColumn("dow_sin", F.sin("dow_angle"))
    .withColumn("dow_cos", F.cos("dow_angle"))

    # month: 1..12
    .withColumn("month_idx", F.col("month") - F.lit(1))
    .withColumn("month_angle", F.lit(2.0 * PI) * F.col("month_idx") / F.lit(12.0))
    .withColumn("month_sin", F.sin("month_angle"))
    .withColumn("month_cos", F.cos("month_angle"))
)

# ------------------------------------------------------------
# 7) TARGET: NET FLOW
# ------------------------------------------------------------
df = df.withColumn(
    "net_flow",
    (F.col("arrivals") - F.col("departures")).cast("double")
)

df = df.withColumn(
    "abs_net_flow",
    F.abs(F.col("net_flow"))
)

# ------------------------------------------------------------
# 8) WINDOW BY STATION
# ------------------------------------------------------------
w = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour"))

# ------------------------------------------------------------
# 9) LAGS FOR NET FLOW
# ------------------------------------------------------------
df = (
    df
    .withColumn("lag1_net",   F.lag("net_flow", 1).over(w))
    .withColumn("lag2_net",   F.lag("net_flow", 2).over(w))
    .withColumn("lag24_net",  F.lag("net_flow", 24).over(w))
    .withColumn("lag168_net", F.lag("net_flow", 168).over(w))
)

# ------------------------------------------------------------
# 10) ROLLING FEATURES FOR NET FLOW
# ------------------------------------------------------------
roll_w_3h  = w.rowsBetween(-3,  -1)
roll_w_24h = w.rowsBetween(-24, -1)

df = (
    df
    .withColumn("roll_mean_3h_net", F.avg("net_flow").over(roll_w_3h))
    .withColumn("roll_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
)

# ------------------------------------------------------------
# 10.1) STATION RECENT ACTIVITY FEATURES
# ------------------------------------------------------------
df = (
    df
    .withColumn("station_mean_24h_net", F.avg("net_flow").over(roll_w_24h))
    .withColumn("station_abs_mean_24h_net", F.avg("abs_net_flow").over(roll_w_24h))
    .withColumn("station_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
)

# ------------------------------------------------------------
# 11) OPTIONAL SANITY CHECKS
# ------------------------------------------------------------
print("Schema after netflow feature engineering:")
df.printSchema()

# ------------------------------------------------------------
# 12) DROP ROWS WITHOUT SUFFICIENT HISTORY
# ------------------------------------------------------------
required_feature_cols = [
    "lag1_net",
    "lag2_net",
    "lag24_net",
    "lag168_net",
    "roll_mean_3h_net",
    "roll_std_24h_net",
    "station_mean_24h_net",
    "station_abs_mean_24h_net",
    "station_std_24h_net",
    "temperature_2m_celsius",
    "apparent_temperature_celsius",
    "event_day_flag",
    "event_day_attendance_sum",
    "events_day_count",
    "event_active_nearby_flag",
    "events_nearby_count",
    "nearest_event_km",
    "event_weighted_intensity",
    "event_attendance_est_sum_nearby",
    "event_impact_score",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

df_feat = df.dropna(subset=required_feature_cols + ["net_flow"])

rows_before_drop = rows_count
rows_after_drop = df_feat.count()

print("Rows before dropna:", f"{rows_before_drop:,}")
print("Rows after dropna :", f"{rows_after_drop:,}")

# ------------------------------------------------------------
# 13) WRITE NEW VERSIONED FEATURE STORE
# ------------------------------------------------------------
dbutils.fs.rm(FEATURES_NETFLOW_DIR, True)

(
    df_feat
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(FEATURES_NETFLOW_DIR)
)

print("✅ Netflow features written to:", FEATURES_NETFLOW_DIR)

# ------------------------------------------------------------
# 14) QUICK VALIDATION
# ------------------------------------------------------------
df_check = spark.read.parquet(FEATURES_NETFLOW_DIR)

print("✅ Validation read successful")
print("Rows written:", f"{df_check.count():,}")
print("Distinct stations:", df_check.select("station_id").distinct().count())

display(
    df_check.select(
        "station_id", "year", "month", "day", "hour",
        "departures", "arrivals", "net_flow",
        "lag1_net", "lag2_net", "lag24_net", "lag168_net",
        "roll_mean_3h_net", "roll_std_24h_net",
        "station_mean_24h_net", "station_abs_mean_24h_net", "station_std_24h_net",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
        "temperature_2m_celsius", "event_active_nearby_flag"
    ).limit(20)
)

## Output

This notebook generates the following feature store:

Path: `goldv2_features_netflow_v3_cyclic_stationtrend`

### Dataset Characteristics:
- Granularity: **station-hour level**
- Coverage: **Downtown stations only**
- Total rows: ~2.36 million
- Stations: 228

### Feature Groups:

**Target Variable**
- `net_flow`

**Lag Features**
- `lag1_net`, `lag2_net`, `lag24_net`, `lag168_net`

**Rolling Features**
- `roll_mean_3h_net`
- `roll_std_24h_net`

**Station Behavior Features**
- `station_mean_24h_net`
- `station_abs_mean_24h_net`
- `station_std_24h_net`

**Cyclic Time Features**
- `hour_sin`, `hour_cos`
- `dow_sin`, `dow_cos`
- `month_sin`, `month_cos`

**External Drivers**
- Weather variables
- Event-related features

This dataset is used directly by:
- 1-hour prediction pipeline (Job ABC)
- Multi-hour forecasting engine (Job D)

## Key Insights & Interpretation

### 1. Strong Temporal Patterns Captured
The use of cyclic features (sin/cos) allows the model to learn:
- Daily commuting cycles
- Weekly behavioral patterns
- Seasonal effects

This avoids artificial discontinuities (e.g., hour 23 → 0).

---

### 2. Recent Behavior is Highly Predictive
Lag and rolling features capture:
- Momentum in station usage
- Short-term demand fluctuations
- Recurrent patterns (daily and weekly)

These are critical for short-term forecasting accuracy.

---

### 3. Station-Specific Dynamics Matter
Station-level statistics (mean, variability) enable the model to:
- Differentiate between high-demand and low-demand stations
- Capture localized patterns
- Improve generalization across stations

---

### 4. External Factors Add Context
Weather and event features introduce:
- Demand shocks (events)
- Environmental influence (temperature)

This enhances model robustness under real-world variability.

---

### 5. Data Quality and Consistency
After filtering incomplete records:
- ~98% of data retained
- No nulls in critical features
- Stable dataset for ML training

---

### 6. Business Impact
This feature layer enables:
- Accurate short-term predictions (1h ahead)
- Reliable multi-hour forecasts (up to 7 days)
- Operational decision-making (bike rebalancing, risk alerts)

It represents the **core feature layer** of the forecasting system and provides the model-ready dataset used by the downstream training, evaluation, and serving pipelines.